# Phase 1
In Anti-Money Laundering (AML), especially in cryptocurrency, criminals don't just steal money and keep it still. They move it through complex webs of accounts to "clean" it (a process called layering). To catch them, we cannot look at transactions in a vacuum; we must look at the relationships between them.

This brings us to Graph Networks. In a graph:
- Nodes are entities (in our case, Bitcoin transactions).
- Edges are the connections between them (the flow of Bitcoin from one transaction to the next).
The goal of this phase is to built a model that catches illicit transactions not just based on the transaction's own features, but based on who it interacts with in the network.

In [1]:
import pandas as pd
from pathlib import Path

The Elliptic dataset is the world's largest labeled dataset of cryptocurrency transactions. It contains over 200,000 Bitcoin transactions.
- Some are labeled as "illicit" (tied to scams, malware, terrorist organizations, etc.).
- Some are labeled as "licit" (exchanges, wallet providers, miners).
- The rest are "unknown.

In [ ]:
import kaggle

SyntaxError: invalid syntax (4193469513.py, line 2)

In [ ]:
# Download the dataset using Kaggle API with mac
!kaggle datasets download ellipticco/elliptic-data-set

Dataset URL: https://www.kaggle.com/datasets/ellipticco/elliptic-data-set
License(s): Attribution-NonCommercial-NoDerivatives 4.0 International (CC BY-NC-ND 4.0)
100%|████████████████████████████████████████| 146M/146M [00:09<00:00, 17.0MB/s]



### Wrangling and exploration

In [10]:
#labels
# unknown, 2 (licit), and 1 (illicit).
df_classes = pd.read_csv(Path("elliptic_bitcoin_dataset") / "elliptic_txs_classes.csv")
# two columns of source and destination
df_edgelist = pd.read_csv(Path("elliptic_bitcoin_dataset") / "elliptic_txs_edgelist.csv")
# 165 anonymised mathematical features of every transaction, and a time step from 1 to 49 representing two week intervals. no column names for the features file, so we set header=None
df_features = pd.read_csv(Path("elliptic_bitcoin_dataset") / "elliptic_txs_features.csv", header=None)

print("Classes Overview")
print(df_classes.head())
print("\nClass distribution")
print(df_classes["class"].value_counts())


Classes Overview
        txId    class
0  230425980  unknown
1    5530458  unknown
2  232022460  unknown
3  232438397        2
4  230460314  unknown

Class distribution
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64


**Observation:** 
77% of our data (157,205 transactions) is "unknown". This reflects the harsh reality of AML: investigators simply don't have the time to investigate and label every single Bitcoin transaction. We only know the true labels for a small fraction of the network. This pushes us into the realm of Semi-Supervised Learning or Node Classification.

For now, to train our initial models, we will only use the transactions we know the answers to (the 1s and 2s). Later, we can use the graph structure to let the "bad" reputation of known illicit nodes bleed over into the "unknown" nodes they interact with.

In [ ]:
# merge data labels (df_classes) and our math features (df_features)
# standard ML conventions: unknown -> -1, licit -> 0, ilicit -> 1
class_mapping = {'unknown': -1, '1': 1, '2': 0}
df_classes['class'] = df_classes['class'].map(class_mapping)

# 2. Prepare the features dataframe for merging
# In df_features, column 0 is the txId, column 1 is the time step. 
# We need to name column 0 'txId' so pandas knows how to match the two tables.
print(df_features.head())
df_features = df_features.rename(columns={0: 'txId', 1: 'time_step'})
print(df_features.head())

         0    1         2         3         4          5         6    \
0  230425980    1 -0.171469 -0.184668 -1.201369  -0.121970 -0.043875   
1    5530458    1 -0.171484 -0.184668 -1.201369  -0.121970 -0.043875   
2  232022460    1 -0.172107 -0.184668 -1.201369  -0.121970 -0.043875   
3  232438397    1  0.163054  1.963790 -0.646376  12.409294 -0.063725   
4  230460314    1  1.011523 -0.081127 -1.201369   1.153668  0.333276   

        7          8         9    ...       157       158       159       160  \
0 -0.113002  -0.061584 -0.162097  ... -0.562153 -0.600999  1.461330  1.461369   
1 -0.113002  -0.061584 -0.162112  ...  0.947382  0.673103 -0.979074 -0.978556   
2 -0.113002  -0.061584 -0.162749  ...  0.670883  0.439728 -0.979074 -0.978556   
3  9.782742  12.414558 -0.163645  ... -0.577099 -0.613614  0.241128  0.241406   
4  1.312656  -0.061584 -0.163523  ... -0.511871 -0.400422  0.517257  0.579382   

        161       162       163       164       165       166  
0  0.018279 -0.0

In [13]:
df = pd.merge(df_classes, df_features, on='txId')
print("Merged dataset shape:", df.shape)
print("Preview of the first 5 columns:")
print(df.iloc[:5, :5])

Merged dataset shape: (203769, 168)
Preview of the first 5 columns:
        txId  class  time_step         2         3
0  230425980     -1          1 -0.171469 -0.184668
1    5530458     -1          1 -0.171484 -0.184668
2  232022460     -1          1 -0.172107 -0.184668
3  232438397      0          1  0.163054  1.963790
4  230460314     -1          1  1.011523 -0.081127


Each time_step represents roughly a two-week period.
- time_step 1 contains all the transactions from the first two weeks of their data collection.
- time_step 2 contains transactions from the next two weeks... all the way up to time_step 49. <br>

If we randomly shuffle the Elliptic Data Set, we fall into a massive trap called Data Leakage (specifically, "look-ahead bias").

### Building a standard model as a baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1. Filter out the 'unknown' transactions (-1) for our baseline
df_labeled = df[df['class'] != -1]

# 2. Temporal Split (Train on past, Test on future) - avoiding look ahead bias (rather than an 80/20 random split)
train_df = df_labeled[df_labeled['time_step'] <= 34]
test_df = df_labeled[df_labeled['time_step'] > 34]

# 3. Separate features (X) and labels (y)
# We drop 'txId', 'class', and 'time_step' from our features because they aren't math features
cols_to_drop = ['txId', 'class', 'time_step']

X_train = train_df.drop(columns=cols_to_drop)
y_train = train_df['class']

X_test = test_df.drop(columns=cols_to_drop)
y_test = test_df['class']

print(f"Training on {len(X_train)} transactions, testing on {len(X_test)} transactions.")

# 4. Train a Baseline Random Forest
print("Training Baseline Random Forest...")
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_baseline.fit(X_train, y_train)

# 5. Evaluate
y_pred = rf_baseline.predict(X_test)
print("\n--- Baseline Model Report ---")
print(classification_report(y_test, y_pred))